# MCP Servers with FastMCP


The **Model Context Protocol** (MCP) is an open protocol that lets LLM clients discover and call external tools, read resources, and receive structured data through a uniform interface. Rather than every tool vendor shipping a custom plugin format for each AI platform, MCP defines a single contract: servers expose a list of tools and resources; clients call them. Any MCP-compatible client — OpenCode, Claude Desktop, Cursor, VS Code Copilot — can connect to any MCP-compatible server without integration work on either side.


**FastMCP** is the de facto Python library for building MCP servers. It handles protocol serialization, schema generation, transport negotiation, and type validation, so we write plain Python functions and FastMCP turns them into callable tools. FastMCP 1.0 was incorporated into the official MCP Python SDK, and the library now powers roughly $70$% of all MCP servers in the ecosystem. This notebook covers the full workflow: building a server with `@mcp.tool` and `@mcp.resource`, testing it with the in-process client, wiring it into OpenCode, and building a concrete project-specific server for `ai-notebooks`.


**Install.** FastMCP is available on PyPI:

```{.bash filename="$ (local)"}
uv add fastmcp
```


## FastMCP Servers


**Minimal server.** A FastMCP server is an instance of `FastMCP`. We instantiate it with a display name and decorate functions with `@mcp.tool` to register them:

```{.python filename="server.py"}
from fastmcp import FastMCP

mcp = FastMCP("My Server")

@mcp.tool
def greet(name: str) -> str:
    """Return a greeting for the given name."""
    return f"Hello, {name}!"

if __name__ == "__main__":
    mcp.run()
```

FastMCP infers the tool name from the function name, the tool description from the docstring, and the parameter schema from the type annotations. No schema boilerplate is needed.


**Type annotations and schema generation.** FastMCP supports the full range of Python and Pydantic types as parameters: `int`, `float`, `str`, `bool`, `list[T]`, `dict[K, V]`, `T | None`, `Literal[...]`, `Enum`, Pydantic models, and dataclasses. Required parameters have no default value; optional parameters have defaults:

```python
from typing import Annotated

@mcp.tool
def search(
    query: Annotated[str, "Full-text search query"],
    limit: Annotated[int, "Maximum number of results"] = 10,
    tag: str | None = None,
) -> list[dict]:
    """Search the database."""
    ...
```

The `Annotated[T, "description"]` shorthand attaches a human-readable description to each parameter — the LLM sees these descriptions when deciding how to call the tool.


**Async tools.** We can define tools as `async def` for I/O-bound operations. Synchronous tools are automatically run in a thread pool so they don't block the event loop, but async is preferred when calling network APIs or databases:

```python
import httpx

@mcp.tool
async def fetch_url(url: str) -> str:
    """Fetch the text content at a URL."""
    async with httpx.AsyncClient() as client:
        resp = await client.get(url, follow_redirects=True)
        resp.raise_for_status()
        return resp.text
```


**Resources.** Resources are read-only data sources identified by a URI. The LLM can read them to retrieve file contents, configuration, or any contextual data. We register them with `@mcp.resource`:

```python
import json

@mcp.resource("data://config")
def get_config() -> str:
    """Return the current application configuration."""
    return json.dumps({"version": "1.0", "debug": False})
```

URI placeholders create **resource templates** — the function is called with the extracted parameter for each requested URI:

```python
@mcp.resource("notebook://{name}")
def get_notebook_source(name: str) -> str:
    """Return the raw JSON source of a notebook file."""
    path = NOTEBOOKS_DIR / name
    return path.read_text()
```


**Running the server.** `mcp.run()` defaults to the `stdio` transport — the server reads from `stdin` and writes to `stdout`, which is how local MCP clients (including OpenCode) spawn it as a subprocess. For remote access we pass `transport="http"`:

```python
if __name__ == "__main__":
    mcp.run()                              # stdio (local)
    # mcp.run(transport="http", port=8000) # HTTP (remote)
```

Alternatively, the FastMCP CLI runs any server object directly without executing the `__main__` block:

```{.bash filename="$ (local)"}
fastmcp run server.py:mcp
fastmcp run server.py:mcp --transport http --port 8000
```


## FastMCP Client


FastMCP ships a `Client` class for programmatic interaction with any MCP server. All operations are async and require an `async with client:` context manager that handles the connection lifecycle and initialization handshake.


**Transport selection.** The client infers the appropriate transport from what we pass to it:

```python
from fastmcp import Client, FastMCP

# In-memory (same process) — ideal for notebooks and tests
server = FastMCP("TestServer")
client = Client(server)

# HTTP remote server
client = Client("https://mcp.context7.com/mcp")

# Local Python file — spawned as a subprocess
client = Client("server.py")
```

The **in-memory** transport connects directly to a `FastMCP` instance in the same Python process. It eliminates subprocess and network overhead, making it the preferred choice for interactive notebook development.


**Listing and calling tools.** `list_tools()` returns the server's tool catalog; `call_tool(name, args)` executes a tool and returns its result:

```python
import asyncio
from fastmcp import Client, FastMCP

mcp = FastMCP("Demo")

@mcp.tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b

async def main():
    async with Client(mcp) as client:
        tools = await client.list_tools()
        print([t.name for t in tools])      # ['add']

        result = await client.call_tool("add", {"a": 3, "b": 4})
        print(result.data)                  # 7

asyncio.run(main())
```


**Reading resources.** `list_resources()` enumerates available resources; `read_resource(uri)` fetches one by URI:

```python
async with Client(mcp) as client:
    resources = await client.list_resources()
    content = await client.read_resource("data://config")
    print(content[0].text)
```


:::{.callout-tip}
In Jupyter notebooks, use `await` directly inside a cell without `asyncio.run()`. The kernel already runs an event loop, so `async with Client(mcp) as client: ...` works at the top level. The `asyncio.run()` wrapper is only needed in plain Python scripts.

:::


## Connecting Servers to OpenCode


OpenCode discovers MCP servers through the `mcp` key in `opencode.json`. Each entry is a named server with a `type` and connection details. Once registered, the server's tools are available to the LLM alongside built-in tools — no special invocation is needed.


**Local servers.** A local server runs as a child subprocess communicating over stdio. We specify `type: "local"` and a `command` array:

```json
{
  "mcp": {
    "my-server": {
      "type": "local",
      "command": ["python", ".opencode/servers/notebooks.py"]
    }
  }
}
```

An optional `environment` object passes environment variables to the subprocess — useful for API keys the server needs:

```json
{
  "mcp": {
    "my-server": {
      "type": "local",
      "command": ["python", ".opencode/servers/notebooks.py"],
      "environment": {
        "SOME_API_KEY": "{env:SOME_API_KEY}"
      }
    }
  }
}
```


**Remote servers.** Remote servers are reached over HTTP with `type: "remote"` and a `url`. Authentication headers can be passed directly or via `{env:VAR}` substitution:

```json
{
  "mcp": {
    "context7": {
      "type": "remote",
      "url": "https://mcp.context7.com/mcp"
    },
    "grep": {
      "type": "remote",
      "url": "https://mcp.grep.app"
    }
  }
}
```

For servers that use OAuth (e.g. Sentry), OpenCode initiates the OAuth flow automatically when it receives a 401. We can also trigger it manually:

```{.bash filename="$ (local)"}
opencode mcp auth sentry
```

Tokens are cached in `~/.local/share/opencode/mcp-auth.json`.


**Notable public servers.** Three servers worth knowing:

| Server | Config key | What it provides |
| :-- | :-- | :-- |
| [Context7](https://github.com/upstash/context7) | `context7` | Up-to-date library documentation resolved from package names |
| [Grep by Vercel](https://grep.app) | `grep` | Full-text code search across GitHub public repositories |
| [Sentry](https://sentry.io) | `sentry` | Access to Sentry issues, events, and stack traces (OAuth) |

Context7 is particularly useful in this project: adding `use context7` to a prompt (or to `AGENTS.md`) lets the agent resolve library names to current documentation snippets rather than relying on potentially stale training data.


**Managing tools per agent.** MCP tools are prefixed with their server name (e.g. `context7_resolve-library-id`). We can scope them to specific agents with the `tools` config key and glob patterns — disable globally, enable only where useful:

```json
{
  "mcp": {
    "context7": { "type": "remote", "url": "https://mcp.context7.com/mcp" }
  },
  "tools": {
    "context7_*": false
  },
  "agent": {
    "notebook-writer": {
      "tools": { "context7_*": true }
    }
  }
}
```

This keeps the default Build agent's context lean while making documentation lookup available to the `notebook-writer` subagent exactly when it is writing new content.


:::{.callout-caution}
Each MCP server adds its full tool list to the context window. A large server can push sessions over the model's context limit. Only enable servers you actively use, and prefer scoping them per agent rather than enabling globally.

:::


## Worked Example: A Project-Specific MCP Server


We build a small MCP server that gives OpenCode structured access to the notebooks in this repository. The built-in tools (`glob`, `bash`, `read`) can already do this — the point of a dedicated server is to provide a **higher-level interface** with domain knowledge baked in: it knows the notebooks directory, understands execution-count cell references, and wraps ripgrep with notebook-aware output.


**The server file.** We place the server at `.opencode/servers/notebooks.py`:

```{.python filename=".opencode/servers/notebooks.py"}
import json
import subprocess
from pathlib import Path
from typing import Annotated

from fastmcp import FastMCP

NOTEBOOKS_DIR = Path(__file__).parent.parent.parent / "notebooks"

mcp = FastMCP("ai-notebooks")


@mcp.tool
def list_notebooks() -> list[str]:
    """Return all notebook paths relative to the notebooks/ directory."""
    return sorted(
        str(p.relative_to(NOTEBOOKS_DIR))
        for p in NOTEBOOKS_DIR.rglob("*.ipynb")
    )


@mcp.tool
def get_cell(
    notebook: Annotated[str, "Notebook path relative to notebooks/, e.g. deep/03.ipynb"],
    cell_n: Annotated[int, "Cell execution count (the number shown in [N] brackets)"],
) -> str:
    """Return the source of the cell with the given execution count."""
    path = NOTEBOOKS_DIR / notebook
    nb = json.loads(path.read_text())
    for cell in nb["cells"]:
        if cell.get("execution_count") == cell_n:
            return "".join(cell["source"])
    raise ValueError(f"No cell with execution_count={cell_n} in {notebook}")


@mcp.tool
def search_notebooks(
    query: Annotated[str, "Regex or literal pattern to search for"],
    section: Annotated[str, "Subdirectory to restrict search, e.g. 'deep' or 'agents'"] = "",
) -> str:
    """Search notebook source cells using ripgrep. Returns matching file:line excerpts."""
    target = NOTEBOOKS_DIR / section if section else NOTEBOOKS_DIR
    result = subprocess.run(
        ["rg", "--type", "json", "-l", query, str(target)],
        capture_output=True,
        text=True,
    )
    return result.stdout.strip() or "No matches found."


if __name__ == "__main__":
    mcp.run()
```


**Wiring it into OpenCode.** We add the server to `opencode.json`:

```json
{
  "$schema": "https://opencode.ai/config.json",
  "instructions": [".opencode/preferences.md", ".github/AI_PREFERENCES.md"],
  "mcp": {
    "notebooks": {
      "type": "local",
      "command": ["python", ".opencode/servers/notebooks.py"]
    }
  }
}
```

OpenCode spawns the subprocess once per session. From that point, tools `notebooks_list_notebooks`, `notebooks_get_cell`, and `notebooks_search_notebooks` are available to the LLM.


**Testing with the in-process client.** Before wiring into OpenCode we can exercise the server directly in a notebook cell — no subprocess, no transport layer:

```python
import sys
sys.path.insert(0, ".")

from fastmcp import Client
from opencode.servers.notebooks import mcp  # import the FastMCP instance

async with Client(mcp) as client:
    tools = await client.list_tools()
    print([t.name for t in tools])
    # ['list_notebooks', 'get_cell', 'search_notebooks']

    nbs = await client.call_tool("list_notebooks", {})
    print(nbs.data[:3])
    # ['agents/01.ipynb', 'agents/02.ipynb', 'agents/patterns/01.ipynb']

    hits = await client.call_tool("search_notebooks", {"query": "CrossEntropyLoss", "section": "deep"})
    print(hits.data)
```


**Using it in a session.** With the server wired in, we can prompt OpenCode directly in natural language:

```
Using the notebooks server, find all notebooks in the deep/ section that mention
"CrossEntropyLoss", then show me the first matching cell from each one.
```

OpenCode will call `notebooks_search_notebooks` to identify the files, then `notebooks_get_cell` for each match — exactly as if a human had `rg`'d the repo and read the relevant cells, but without the manual steps.


:::{.callout-note}
The server above uses `rg --type json` to restrict search to `.json` files (notebooks are stored as JSON). Ripgrep's `--type json` glob is `*.json`, which catches `.ipynb` files since they are JSON. To restrict to `.ipynb` specifically, use `rg -g '*.ipynb'` instead.

:::


---


■
